In [11]:
import os
import shutil
import glob
from google.colab import drive

print("🔵 BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & DỮ LIỆU")
print("-----------------------------------------")

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Xóa code cũ & Clone lại (Đảm bảo sạch sẽ)
os.chdir('/content') # Ra ngoài trước khi xóa
if os.path.exists("/content/IDPS_SOAR_MultiAgent_System"):
    shutil.rmtree("/content/IDPS_SOAR_MultiAgent_System")

print("[-] Đang tải code từ Github (Nhánh Binary_Classification)...")
!git clone https://github.com/TheNam1sOut/IDPS_SOAR_MultiAgent_System.git
os.chdir("/content/IDPS_SOAR_MultiAgent_System")
!git checkout Binary_Classification

# 3. Tìm & Copy dữ liệu từ Drive
print("[-] Đang lấy dữ liệu từ Drive...")
dest_dir = "./data/MachineLearningCSV/MachineLearningCVE"
os.makedirs(dest_dir, exist_ok=True)

# Quét tìm file CSV trong thư mục DoAn
found_files = glob.glob("/content/drive/MyDrive/DoAn/**/*WorkingHours*.csv", recursive=True)
if not found_files:
    found_files = glob.glob("/content/drive/MyDrive/DoAn/**/*.csv", recursive=True)

if found_files:
    for f in found_files:
        shutil.copy(f, dest_dir)
    print(f"✅ Đã chuẩn bị xong {len(found_files)} file dữ liệu.")
else:
    print("❌ LỖI: Không tìm thấy file CSV nào trong Drive/DoAn. Hãy kiểm tra lại.")

🔵 BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & DỮ LIỆU
-----------------------------------------
[-] Đang tải code từ Github (Nhánh Binary_Classification)...
Cloning into 'IDPS_SOAR_MultiAgent_System'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 50 (delta 6), reused 49 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 587.60 KiB | 2.58 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Branch 'Binary_Classification' set up to track remote branch 'Binary_Classification' from 'origin'.
Switched to a new branch 'Binary_Classification'
[-] Đang lấy dữ liệu từ Drive...
✅ Đã chuẩn bị xong 7 file dữ liệu.


In [12]:
print("🔵 BƯỚC 2: CẬP NHẬT CODE (PHƯƠNG ÁN DENSE NETWORK - CHẮC ĂN)")
print("------------------------------------------------------------")

# ==============================================================================
# 1. FILE PREPROCESS (GIỮ NGUYÊN)
# ==============================================================================
code_preprocess = """
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import os
import gc
INPUT_CSV_TRAIN = "./data/split/train.csv"
OUTPUT_CSV_TRAIN = "./data/preprocessed/train_processed.csv"
INPUT_CSV_TEST  = "./data/split/test.csv"
OUTPUT_CSV_TEST = "./data/preprocessed/test_processed.csv"
LABEL_COL = "Label"
SCALER_PATH = "./models/standard_scaler.joblib"
DROP_COLUMNS = ["Bwd PSH Flags", "Bwd URG Flags", "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Fwd Header Length.1", "Subflow Fwd Packets", "Subflow Fwd Bytes", "Subflow Bwd Packets", "Subflow Bwd Bytes", "Avg Bwd Segment Size"]
def clean_data(df):
    df.drop(columns=DROP_COLUMNS, errors="ignore", inplace=True)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df.fillna(0, inplace=True)
    for col in numeric_cols:
        if col != LABEL_COL: df[col] = df[col].astype(np.float32)
    return df
def main():
    os.makedirs("./data/preprocessed", exist_ok=True)
    os.makedirs("./models", exist_ok=True)
    if os.path.exists(INPUT_CSV_TRAIN):
        print("Processing Train...")
        df = pd.read_csv(INPUT_CSV_TRAIN)
        df = clean_data(df)
        scaler = StandardScaler()
        X = df.drop(columns=[LABEL_COL])
        y = df[LABEL_COL]
        X_scaled = scaler.fit_transform(X)
        joblib.dump(scaler, SCALER_PATH)
        pd.concat([pd.DataFrame(X_scaled, columns=X.columns), y], axis=1).to_csv(OUTPUT_CSV_TRAIN, index=False)
        del df, X, X_scaled, y
        gc.collect()
    if os.path.exists(INPUT_CSV_TEST):
        print("Processing Test...")
        df = pd.read_csv(INPUT_CSV_TEST)
        df = clean_data(df)
        X = df.drop(columns=[LABEL_COL])
        y = df[LABEL_COL]
        X_scaled = scaler.transform(X)
        pd.concat([pd.DataFrame(X_scaled, columns=X.columns), y], axis=1).to_csv(OUTPUT_CSV_TEST, index=False)
if __name__ == "__main__": main()
"""
with open("src/detection/5_preprocess.py", "w") as f: f.write(code_preprocess)

# ==============================================================================
# 2. FILE TRAIN (CHUYỂN SANG DENSE NETWORK + AUTO CLASS WEIGHT)
# ==============================================================================
code_train = """
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils import class_weight
import os

TRAIN_DATA_PATH = "./data/preprocessed/train_processed.csv"
MODEL_SAVE_PATH = "./models/binary_model.keras"
LABEL_COL = "Label"
BATCH_SIZE = 2048 # Dense nhẹ nên tăng batch size cho nhanh
EPOCHS = 30

def load_data(path):
    print(f"[+] Loading data (ALL into RAM): {path}")
    df = pd.read_csv(path)
    X = df.drop(columns=[LABEL_COL]).values.astype(np.float32)
    y = df[LABEL_COL].values.astype(np.float32)
    return X, y

def build_model(input_dim):
    # Mạng Dense sâu và rộng hơn để học đặc trưng bảng
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),

        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),

        Dense(1, activation="sigmoid")
    ])
    opt = tf.keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])
    return model

def main():
    os.makedirs("./models", exist_ok=True)
    if not os.path.exists(TRAIN_DATA_PATH): return
    X, y = load_data(TRAIN_DATA_PATH)

    # TỰ ĐỘNG TÍNH CLASS WEIGHT (Chuẩn nhất)
    # Nó sẽ tự ép model học lớp Attack dựa trên sự mất cân bằng thực tế
    unique_classes = np.unique(y)
    weights = class_weight.compute_class_weight('balanced', classes=unique_classes, y=y)
    class_weights = dict(enumerate(weights))
    print(f"[+] Computed Class Weights: {class_weights}")

    model = build_model(X.shape[1])

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor="loss")
    ]

    print("[+] Starting Training (Dense Network)...")
    # Train trực tiếp (fit) không qua generator
    model.fit(
        X, y,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights, # Ép cân bằng dữ liệu ở đây
        validation_split=0.1, # Dùng 10% tập train để validate luôn
        verbose=1
    )
    model.save(MODEL_SAVE_PATH)

if __name__ == "__main__": main()
"""
with open("src/detection/7_train_binary.py", "w") as f: f.write(code_train)

# ==============================================================================
# 3. FILE PREDICT (DENSE NETWORK - KHÔNG CẦN SEQUENCE)
# ==============================================================================
code_predict = """
import numpy as np
import pandas as pd
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, f1_score

TEST_CSV = "./data/preprocessed/test_processed.csv"
MODEL_PATH  = "./models/binary_model.keras"
OUTPUT_PRED_CSV = "./results/binary_test_predictions.csv"
LABEL_COL = "Label"
BATCH_SIZE = 2048

def main():
    if not os.path.exists(MODEL_PATH): return
    model = load_model(MODEL_PATH)
    df = pd.read_csv(TEST_CSV)
    X = df.drop(columns=[LABEL_COL]).values
    y_true = df[LABEL_COL].values

    print("[+] Predicting (Dense)...")
    y_prob = model.predict(X, batch_size=BATCH_SIZE, verbose=1).flatten()

    print("\\n[+] Tuning Threshold...")
    best_thresh, best_f1 = 0.5, 0.0

    # Quét ngưỡng rộng
    thresholds = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9]
    print(f"{'Threshold':<10} | {'Macro F1':<10}")
    print("-" * 25)

    for thresh in thresholds:
        y_pred_temp = (y_prob > thresh).astype(int)
        score = f1_score(y_true, y_pred_temp, average='macro')
        print(f"{thresh:<10.2f} | {score:<10.4f}")
        if score > best_f1:
            best_f1, best_thresh = score, thresh

    print(f"\\n>>> CHỐT NGƯỠNG: {best_thresh}")
    y_pred = (y_prob > best_thresh).astype(int)

    print("========== FINAL REPORT ==========")
    print(classification_report(y_true, y_pred, digits=4, target_names=["Benign", "Attack"]))
    print(confusion_matrix(y_true, y_pred))

    os.makedirs("./results", exist_ok=True)
    pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "prob": y_prob}).to_csv(OUTPUT_PRED_CSV, index=False)

if __name__ == "__main__": main()
"""
with open("src/detection/6_predict.py", "w") as f: f.write(code_predict)

print("✅ Đã chuyển sang mô hình Dense (Nhẹ & Mạnh hơn).")
print("👉 Ông chạy lại Cell số 3 để xem phép màu nhé!")

🔵 BƯỚC 2: CẬP NHẬT CODE (PHƯƠNG ÁN DENSE NETWORK - CHẮC ĂN)
------------------------------------------------------------
✅ Đã chuyển sang mô hình Dense (Nhẹ & Mạnh hơn).
👉 Ông chạy lại Cell số 3 để xem phép màu nhé!


In [13]:
print("🔵 BƯỚC 3: THỰC THI QUY TRÌNH & LƯU MODEL")
print("-----------------------------------------")

# 1. Chạy chuẩn bị dữ liệu
!python src/detection/1_list_label.py
!python src/detection/2_label.py
!python src/detection/4_split.py
!python src/detection/5_preprocess.py

# 2. Train Model
print("\n>>> Đang Train Model...")
!python src/detection/7_train_binary.py

# 3. Predict & Đánh giá
print("\n>>> Đang Predict & Tìm ngưỡng tối ưu...")
!python src/detection/6_predict.py

# 4. Lưu về Drive
print("\n>>> Đang lưu Model về Drive...")
if os.path.exists("./models/binary_model.keras"):
    !cp ./models/binary_model.keras "/content/drive/MyDrive/DoAn/"
    !cp ./models/standard_scaler.joblib "/content/drive/MyDrive/DoAn/"
    !cp ./results/binary_test_predictions.csv "/content/drive/MyDrive/DoAn/"
    print("🎉🎉🎉 XONG! MODEL ĐÃ ĐƯỢC LƯU TRONG DRIVE.")
else:
    print("❌ Lỗi: Không thấy model đâu cả!")

🔵 BƯỚC 3: THỰC THI QUY TRÌNH & LƯU MODEL
-----------------------------------------
Dataset shape: (2138040, 79)
Labels in dataset:
BENIGN
Infiltration
PortScan
DDoS
FTP-Patator
SSH-Patator
Bot
Web Attack - Brute Force
Web Attack - XSS
Web Attack - Sql Injection
[+] Reading ./data/dataset/CleanlyLabelled_Dataset.csv...
[+] Converting to Binary Labels (0: Normal, 1: Attack)...
Label distribution:
Label
0    1833066
1     304974
Name: count, dtype: int64
[+] Saved binary labeled data to ./data/labelled/NumberLabelled.csv
Dataset split completed
Train: (1710432, 79)
Test : (427608, 79)
Processing Train...
Processing Test...

>>> Đang Train Model...
2026-02-05 15:41:40.441996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770306100.463313   12928 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN